# Kaggle timing probe: real GPU throughput for the Qwen2.5-7B ALFWorld replication

**What this notebook does**: measures real wall-clock throughput (seconds/episode,
tokens/episode) for the local-model ALFWorld pipeline on Kaggle's actual GPU, in
three modes -- single GPU, 2-worker (one process per GPU), and vLLM (if it fits) --
then converts each measured throughput into a detectable-effect-size (MDE) estimate
for 1 week and 2 weeks of the 30-GPU-hour/week free quota, using this repo's
`power_formula.py`.

**Nothing here starts the real Kaggle replication run.** This is Task C's timing
probe, prepared locally and run here only to get real numbers -- it does not touch
`results_kaggle/`'s main logging phase.

## Settings to click before running anything

- **Accelerator: GPU T4 x2** (Notebook settings, right sidebar) -- both GPUs are
  needed for the 2-worker cell; the single-GPU and vLLM cells only use GPU 0.
- **Internet: ON** (same sidebar) -- needed to `pip install` and to download the
  model/embedding weights and the ALFWorld dataset.
- **Persistence**: not required for this notebook (it's a short timing probe, not
  the multi-session main run) -- default "No persistence" is fine.

## Which cells to run, in order

1. Section 0 (setup) -- always run.
2. Section 1 (vLLM compatibility probe) -- always run; decides whether Section 2's
   vLLM cell is safe to run at all.
3. Section 2, cell "Mode A: single GPU" -- always run.
4. Section 2, cell "Mode B: 2 workers" -- always run (needs both T4s).
5. Section 2, cell "Mode C: vLLM" -- run ONLY if Section 1 printed `vllm_works = True`.
6. Section 3 (results table) -- always run.
7. Section 4 (parity check) -- run ONLY if Section 1's vLLM probe passed.
8. Section 5 (power formula / MDE) -- always run, after Section 3.

## 0. Setup -- install dependencies and get the code onto this machine

In [ ]:
import os
# Standard fix for the PyTorch CUDA caching allocator's OOM-from-fragmentation
# failure mode -- set before anything imports torch. See README.md's OOM section.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import sys, subprocess, shutil
print('Python:', sys.version)
subprocess.run(['nvidia-smi'])


In [ ]:
# Pulls this project's actual code (not reimplemented here) -- replace with your
# own clone URL/token if the repo is private (Kaggle Secrets -> add a
# GITHUB_TOKEN, then use https://<token>@github.com/... below).
!git clone https://github.com/ibo-agentic/memory-pilot.git
%cd memory-pilot
!git rev-parse HEAD


In [ ]:
!pip install alfworld tqdm termcolor sentence-transformers --quiet
!pip install transformers accelerate bitsandbytes --quiet
!alfworld-download

# Editable-installs both packages so alfworld_pilot and memory_ope import in
# EVERY process (including timing_probe.py's own --workers subprocesses) with
# no PYTHONPATH needed anywhere -- verified locally, see README's Orchestration section.
!pip install -e . --quiet
!pip install -e alfworld_pilot --quiet


In [ ]:
import sys, site, pathlib, importlib

# A Kaggle kernel's sys.path is snapshotted at process start, so a `pip
# install -e` a few cells ago is not guaranteed to be visible yet, and any
# earlier failed/partial import attempt can leave a broken module cached in
# sys.modules -- both observed for real on Kaggle (session 1 needed this
# fixed manually). Re-scan site-packages for new .pth files (this is what
# actually registers an editable install's path), and drop any already-
# cached alfworld_pilot/memory_ope modules so the next import is fresh.
importlib.reload(site)
site_dirs = list(site.getsitepackages())
try:
    site_dirs.append(site.getusersitepackages())
except Exception:
    pass
for _dir in site_dirs:
    if _dir and pathlib.Path(_dir).is_dir():
        site.addsitedir(_dir)

for _mod_name in list(sys.modules):
    if _mod_name == 'alfworld_pilot' or _mod_name.startswith('alfworld_pilot.') or _mod_name == 'memory_ope' or _mod_name.startswith('memory_ope.'):
        del sys.modules[_mod_name]

print('sys.path refreshed and cached alfworld_pilot/memory_ope modules cleared.')


In [ ]:
try:
    import alfworld_pilot
    import memory_ope
    from alfworld_pilot import timing_probe, vllm_client, vllm_parity_check  # noqa: F401
    print('Import check PASSED:')
    print('  alfworld_pilot ->', alfworld_pilot.__file__)
    print('  memory_ope     ->', memory_ope.__file__)
except ImportError as e:
    raise SystemExit(f'Import check FAILED: {e!r} -- fix the install cell above before proceeding.')


## 1. vLLM compatibility probe -- does it actually fit on a T4?

Per `vllm_client.py`'s docstring: AWQ (`Qwen/Qwen2.5-7B-Instruct-AWQ`) is expected to
fit comfortably; full fp16 is flagged as tight/risky. This cell is the actual
empirical answer -- nothing before this point has run on real T4 hardware. It also
prints the exact commit hash actually downloaded for the AWQ repo, which you should
paste into `alfworld_pilot/kaggle_config.yaml`'s `llm_vllm.revision` (currently
`null` -- `VLLMClient` refuses to construct until it's a real pinned hash, same rule
as every other model in this project).

In [ ]:
vllm_works = False
vllm_revision = None
try:
    !pip install vllm --quiet
    from vllm import LLM, SamplingParams
    from huggingface_hub import model_info

    MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct-AWQ'
    info = model_info(MODEL_ID)
    vllm_revision = info.sha
    print('Resolved revision for', MODEL_ID, ':', vllm_revision)
    print('-> paste this into alfworld_pilot/kaggle_config.yaml (llm_vllm.revision) before using the vLLM backend for anything beyond this probe.')

    llm = LLM(model=MODEL_ID, revision=vllm_revision, dtype='half', quantization='awq',
              gpu_memory_utilization=0.85, max_model_len=2048)
    out = llm.chat([[{'role': 'user', 'content': 'Say hello in one short sentence.'}]],
                   SamplingParams(temperature=0, max_tokens=20), use_tqdm=False)
    print('vLLM AWQ PASS:', out[0].outputs[0].text)
    vllm_works = True
    del llm
    import torch, gc
    gc.collect(); torch.cuda.empty_cache()
except Exception as e:
    print('vLLM (AWQ) did not work cleanly on this GPU. Error was:')
    print(repr(e))

print()
print('vllm_works =', vllm_works)


## 2. Timing probe, three modes (20 episodes each)

Each cell writes its own `results_kaggle/timing_probe*.json` (already handled by
`timing_probe.py`'s own file naming for multi-worker mode). Uses `pick_and_place_simple`
at `start_seed=1000` (past the local capability-check's own 0-19 range) so nothing here
replays already-cached responses.

In [ ]:
%cd alfworld_pilot
# Mode A: single GPU, transformers backend
!python -m alfworld_pilot.timing_probe --n-episodes 20 --start-seed 1000 --backend transformers


In [ ]:
# Mode B: 2 workers (one per T4), transformers backend -- needs GPU T4 x2 enabled
!python -m alfworld_pilot.timing_probe --n-episodes 20 --start-seed 2000 --backend transformers --workers 2


In [ ]:
# Mode C: vLLM -- ONLY run this cell if Section 1 printed vllm_works = True
if vllm_works:
    get_ipython().system('python -m alfworld_pilot.timing_probe --n-episodes 20 --start-seed 3000 --backend vllm')
else:
    print('Skipping: vLLM did not pass the Section 1 compatibility probe on this GPU.')


## 3. Combined results table

In [ ]:
import json, pathlib

rows = []

single = json.load(open('results_kaggle/timing_probe.json'))
rows.append({
    'mode': 'single GPU (transformers)',
    'n_episodes': single['summary']['n_episodes'],
    'mean_s_per_episode': single['summary']['mean_seconds_per_episode'],
    'median_s_per_episode': single['summary']['median_seconds_per_episode'],
    'episodes_per_30gpuhr_mean': single['summary']['episodes_per_30_gpu_hours_mean'],
})

combined = json.load(open('results_kaggle/timing_probe_combined.json'))
rows.append({
    'mode': '2 workers (transformers)',
    'n_episodes': combined['total_episodes'],
    'mean_s_per_episode': combined['mean_seconds_per_episode'],
    'median_s_per_episode': combined['median_seconds_per_episode'],
    'episodes_per_30gpuhr_mean': combined['combined_episodes_per_30_gpu_hours'],
})

if vllm_works:
    # Mode C overwrites the same default output path Mode A used earlier in this
    # session -- read fresh here since Mode C ran most recently. If you re-run
    # cells out of order, rename Mode A's output file first.
    vllm_report = json.load(open('results_kaggle/timing_probe.json'))
    if vllm_report.get('backend') == 'vllm':
        rows.append({
            'mode': 'single GPU (vLLM AWQ)',
            'n_episodes': vllm_report['summary']['n_episodes'],
            'mean_s_per_episode': vllm_report['summary']['mean_seconds_per_episode'],
            'median_s_per_episode': vllm_report['summary']['median_seconds_per_episode'],
            'episodes_per_30gpuhr_mean': vllm_report['summary']['episodes_per_30_gpu_hours_mean'],
        })
    else:
        print('WARNING: results_kaggle/timing_probe.json is not a vLLM report -- rerun Mode C or check file paths.')

import pandas as pd
df = pd.DataFrame(rows)
print(df.to_string(index=False))


## 4. Backend parity check (only if vLLM ran)

Same task_ids (seeds) on both backends -- checks the PARSED ACTION sequence matches,
not raw text (see `vllm_parity_check.py`'s docstring for why raw-text equality isn't
the right bar across backends).

In [ ]:
if vllm_works:
    get_ipython().system('python -m alfworld_pilot.vllm_parity_check --n-episodes 5')
else:
    print('Skipping: vLLM was not available.')


## 5. Detectable effect size (MDE) at 1 week and 2 weeks, per measured throughput

Uses this repo's own `power_formula.py` (paired forced-in/forced-out design), with
`n_memories=6` (this project's provisional ground-truth subset size) and `p=0.456`
(the Task B expected pooled success rate under the new task-type weights). `rho=0.6355`
is still borrowed from the paid run's measured GPT-5.6-Luna correlation -- not yet
measured for Qwen -- exactly as flagged in `alfworld_pilot/README.md`'s Timing
estimate section; this table should be re-derived once a real Qwen `rho` exists.

In [ ]:
from memory_ope.evaluation.power_formula import var_of_paired_difference
from scipy.stats import norm
import math

p = 0.456
rho = 0.6355  # borrowed from the paid run -- see caveat above
n_memories = 6
alpha, power = 0.05, 0.80
z_a = norm.ppf(1 - alpha / 2)
z_b = norm.ppf(power)
sigma_d2 = var_of_paired_difference(p, rho)

def mde_for_n_episodes(n_total_episodes):
    n_pairs = n_total_episodes / (2 * n_memories)
    return (z_a + z_b) * math.sqrt(sigma_d2 / n_pairs)

header = f"{'mode':30s} {'s/episode (mean)':>18s} {'N in 1 week':>14s} {'MDE 1wk':>9s} {'N in 2 weeks':>14s} {'MDE 2wk':>9s}"
print(header)
for r in rows:
    spe = r['mean_s_per_episode']
    n_1wk = (30 * 3600) / spe
    n_2wk = (60 * 3600) / spe
    line = f"{r['mode']:30s} {spe:18.2f} {n_1wk:14.0f} {mde_for_n_episodes(n_1wk):9.4f} {n_2wk:14.0f} {mde_for_n_episodes(n_2wk):9.4f}"
    print(line)


## Done

Report back: the Section 3 table, whether Section 1's vLLM probe passed (and the
resolved revision it printed, if so), and the Section 4 parity result (if it ran).
No further Kaggle work should be started from this notebook -- the main replication
run is a separate, much larger, multi-session effort (see `README.md`'s Kaggle
replication section).